Kinesis and MSK are AWS's managed streaming services for ingesting, processing, and analysing real-time data at scale. Kinesis is a family of AWS-native services covering data streams, delivery, and analytics. MSK — Managed Streaming for Apache Kafka — is a fully managed Kafka service for teams already using the Kafka ecosystem. Both appear on the SAA-C03 exam, with Kinesis covered in more depth.

## Streaming vs Batch

```text
Batch (SQS / scheduled jobs):     Streaming (Kinesis):
Events accumulate → process later  Events processed as they arrive
Latency: minutes to hours          Latency: milliseconds to seconds
```

Use streaming when you need to act on data **as it arrives**: fraud detection, real-time dashboards, live recommendations, IoT telemetry, log analytics. Use batch when latency tolerance is high and processing in bulk is more efficient.

**Kinesis vs SQS:**
- SQS: work queue — each message processed by one consumer, then deleted
- Kinesis: data stream — records retained for up to 365 days, multiple consumers can read the same data independently

## Kinesis Data Streams (KDS)

Kinesis Data Streams is a real-time data streaming service. Producers push records into a stream; consumers read and process them.

### Shards
A stream is divided into **shards** — the unit of capacity:

| Direction | Capacity per shard |
|---|---|
| **Ingestion (write)** | 1 MB/s or 1,000 records/s |
| **Consumption (read)** | 2 MB/s (shared across all standard consumers) |

Scale by adding shards (**shard splitting**) or reducing them (**shard merging**).

```text
Producers                 Stream (4 shards)          Consumers
App servers ──┐           ┌─ Shard 1 ─┐
IoT devices ──┼──PUT──▶  ├─ Shard 2 ─┤  ──GET──▶  Lambda
Clickstream ──┘           ├─ Shard 3 ─┤            EC2 workers
                          └─ Shard 4 ─┘            Kinesis Analytics
```

### Records
Each record contains:
- **Partition key** — hashed to determine which shard receives the record; use high-cardinality keys to distribute load evenly across shards
- **Sequence number** — assigned by KDS; unique per shard, monotonically increasing
- **Data blob** — up to 1 MB

### Retention
- Default: **24 hours**
- Extended: up to **7 days** (additional cost)
- Long-term: up to **365 days**
- Records remain in the stream after consumption — consumers re-read from any point within the retention window

### Capacity modes
| Mode | How | Use case |
|---|---|---|
| **Provisioned** | Manually specify shard count | Predictable throughput; you manage scaling |
| **On-demand** | Auto-scales capacity | Unpredictable or variable traffic; pay per GB |

## KDS Consumers

### Standard consumers (GetRecords)
- Poll the stream using `GetRecords`
- **2 MB/s read throughput per shard shared across all standard consumers**
- If 5 consumers read from the same shard, each gets 400 KB/s
- Latency: ~200ms
- Up to 5 `GetRecords` calls per shard per second

### Enhanced Fan-Out (EFO)
- Consumers subscribe to the stream using `SubscribeToShard`
- **Each EFO consumer gets its own dedicated 2 MB/s per shard** — not shared
- KDS pushes records to consumers (HTTP/2 push) instead of consumers polling
- Latency: ~70ms
- Use when you have multiple consumers that all need full throughput

```text
Standard: 5 consumers × 1 shard = 400 KB/s each
EFO:      5 consumers × 1 shard = 2 MB/s each  (dedicated per consumer)
```

### Consumer checkpointing
Consumers track their position in the stream using a **sequence number checkpoint** (commonly stored in DynamoDB via the Kinesis Client Library). On restart, the consumer resumes from where it left off.

### Starting positions
| Position | Description |
|---|---|
| `TRIM_HORIZON` | Start from the oldest available record |
| `LATEST` | Start from new records only (skip history) |
| `AT_SEQUENCE_NUMBER` | Resume from a specific record |
| `AT_TIMESTAMP` | Start from a specific point in time |

## Kinesis Data Firehose

Firehose is a fully managed **delivery service** — it loads streaming data into destinations without you managing consumers, scaling, or checkpointing.

### How Firehose works
```text
Producers / KDS stream
        │
        ▼
  Kinesis Firehose
  ┌─ optional Lambda transform ─┐
  └─ buffer (size or time) ─────┘
        │
        ▼
  Destination
```

### Destinations
- **Amazon S3** — primary destination; Firehose can also back up raw records to S3 before transformation
- **Amazon Redshift** — Firehose writes to S3 first, then issues a COPY command
- **Amazon OpenSearch Service** — real-time log/metrics analytics
- **Splunk** — third-party SIEM
- **HTTP endpoint** — any custom destination
- **Datadog / New Relic / Dynatrace / Coralogix**

### Key properties
- **Fully managed** — no shards, no consumers to manage, auto-scales
- **Near real-time** — buffers data before delivering: buffer size (1–128 MB) or buffer interval (60–900 seconds), whichever is reached first
- **Lambda transformation** — optionally invoke a Lambda function to transform records before delivery (e.g., convert JSON to Parquet, filter fields, enrich data)
- **No data storage** — Firehose is a delivery pipe, not a store; failed records can be backed up to S3
- **Data formats** — can convert JSON to Parquet/ORC using the AWS Glue schema

### KDS vs Firehose

| | KDS | Firehose |
|---|---|---|
| Latency | Real-time (ms) | Near real-time (buffer: 60s+) |
| Consumers | Custom (Lambda, EC2, KCL) | Managed delivery to fixed destinations |
| Data retention | 1–365 days | No retention — delivery pipe |
| Scaling | Manual (provisioned) or auto (on-demand) | Fully automatic |
| Use case | Custom real-time processing | Load streaming data to S3/Redshift/OpenSearch |

> **Exam tip:** "Load streaming data to S3 / Redshift / OpenSearch" → Firehose. "Real-time processing with custom logic" → KDS + Lambda/EC2.

## Kinesis Data Analytics

Kinesis Data Analytics runs **real-time analytics** on streaming data using SQL or Apache Flink.

### Two flavours
| | KDA for SQL | KDA for Apache Flink |
|---|---|---|
| Language | SQL | Java, Scala, Python (Flink) |
| Complexity | Simple transformations, aggregations | Complex stateful processing, ML |
| Use case | Real-time metrics, anomaly detection | Advanced stream processing |

### Architecture
```text
KDS or Firehose  →  Kinesis Data Analytics  →  KDS / Firehose / Lambda
   (input)              (SQL / Flink)              (output)
```

**Common use cases:**
- Real-time dashboards: compute rolling averages, counts per minute
- Anomaly detection: flag when a metric exceeds a threshold in the last N seconds
- ETL: filter, enrich, and transform records before landing in S3
- Time-series analytics: windowed aggregations (tumbling, sliding, session windows)

## Amazon MSK — Managed Streaming for Apache Kafka

MSK is a fully managed Apache Kafka service. It provisions and manages Kafka **brokers** and **ZooKeeper** nodes (or KRaft for newer versions), handles patching, and replicates data across Availability Zones.

### Kafka concepts
- **Topic**: a named stream of records (like a Kinesis stream)
- **Partition**: a topic is split into partitions; each partition is an ordered, immutable log. Records with the same key go to the same partition (like Kinesis partition key → shard)
- **Broker**: a Kafka server that stores partitions and serves producers/consumers
- **Producer**: writes records to topics
- **Consumer group**: a group of consumers that collectively read all partitions of a topic — each partition assigned to one consumer in the group
- **Offset**: position of a consumer within a partition (like KDS sequence number)

```text
Topic: orders  (3 partitions)
├── Partition 0: [rec0, rec1, rec4, ...]
├── Partition 1: [rec2, rec5, ...]
└── Partition 2: [rec3, rec6, ...]

Consumer Group A:  Consumer-1 ← P0, Consumer-2 ← P1, Consumer-3 ← P2
Consumer Group B:  reads all partitions independently (full copy)
```

### MSK key properties
- Kafka-compatible — use existing Kafka producers, consumers, and connectors without code changes
- Multi-AZ by default — brokers spread across 2–3 AZs
- Storage: EBS volumes per broker, configurable size
- **MSK Serverless**: no broker management; auto-scales capacity; pay per usage
- **MSK Connect**: run managed Kafka Connect connectors to move data between Kafka and S3, DynamoDB, RDS, etc.
- Data retained based on retention policy (time-based or size-based) — default 7 days

### MSK vs Kinesis

| | Kinesis Data Streams | Amazon MSK |
|---|---|---|
| Protocol | AWS proprietary | Apache Kafka |
| Max message size | 1 MB | 1 MB default (configurable up to 10 MB+) |
| Retention | 1–365 days | Configurable (time or size) |
| Scaling | Shard split/merge | Add/remove brokers + partitions |
| Serverless option | On-demand mode | MSK Serverless |
| Security | IAM, KMS, TLS | IAM, TLS, SASL/SCRAM, mTLS |
| Consumers | AWS SDK / KCL | Kafka client libraries |
| Ecosystem | AWS native | Full Kafka ecosystem (connectors, Kafka Streams, ksqlDB) |
| Use case | New AWS-native streaming | Migrate Kafka workloads; Kafka ecosystem required |

> **Exam tip:** "Apache Kafka" or "migrate existing Kafka" → MSK. Building net-new on AWS → Kinesis.

## Architecture Patterns

### Real-time clickstream analytics
```text
Web/App → Kinesis Data Streams → Kinesis Data Analytics (SQL)
                │                        │
                │                  real-time dashboard
                ▼
          Firehose → S3 (raw archive) → Athena (ad-hoc queries)
```

### IoT pipeline
```text
IoT devices → Kinesis Data Streams
                 ├──▶ Lambda (real-time alerts)
                 └──▶ Firehose → S3 → Glue → Redshift (batch analytics)
```

### Log ingestion to OpenSearch
```text
EC2 / ECS logs → Firehose (optional Lambda transform) → OpenSearch
                                │
                          S3 backup (failed records)
```

### Kinesis vs SQS decision
| Scenario | Use |
|---|---|
| Multiple consumers reading the same data stream | KDS |
| Ordered records per partition key | KDS |
| Replay historical data within retention window | KDS |
| Work queue — one consumer per message | SQS |
| Unknown or variable number of consumers | SQS |
| Load data to S3/Redshift/OpenSearch | Firehose |

## Working with Kinesis via boto3

In [ ]:
import boto3
import json
import base64

kinesis   = boto3.client('kinesis',   region_name='us-east-1')
firehose  = boto3.client('firehose',  region_name='us-east-1')

In [ ]:
# Create a Kinesis Data Stream (provisioned, 4 shards)
kinesis.create_stream(
    StreamName='clickstream',
    ShardCount=4,                # 4 MB/s write, 8 MB/s read capacity
)

# Or on-demand (auto-scaling)
kinesis.create_stream(
    StreamName='events-ondemand',
    StreamModeDetails={'StreamMode': 'ON_DEMAND'}
)
print("Streams created")

In [ ]:
# Put records — batch up to 500 records or 5 MB per call
records = [
    {
        'Data': json.dumps({'userId': 'u-001', 'page': '/checkout', 'ts': 1700000001}),
        'PartitionKey': 'u-001'   # same user → same shard → ordered for this user
    },
    {
        'Data': json.dumps({'userId': 'u-002', 'page': '/home',     'ts': 1700000002}),
        'PartitionKey': 'u-002'
    },
]

response = kinesis.put_records(
    StreamName='clickstream',
    Records=[
        {'Data': r['Data'].encode(), 'PartitionKey': r['PartitionKey']}
        for r in records
    ]
)
print(f"Failed records: {response['FailedRecordCount']}")

In [ ]:
# Read records from a shard (simple consumer)
stream_info = kinesis.describe_stream_summary(StreamName='clickstream')

# Get the first shard
shards = kinesis.list_shards(StreamName='clickstream')['Shards']
shard_id = shards[0]['ShardId']

# Get shard iterator (start from oldest available record)
iterator_response = kinesis.get_shard_iterator(
    StreamName='clickstream',
    ShardId=shard_id,
    ShardIteratorType='TRIM_HORIZON'
)
shard_iterator = iterator_response['ShardIterator']

# Read records
records_response = kinesis.get_records(
    ShardIterator=shard_iterator,
    Limit=10
)

for record in records_response['Records']:
    data = json.loads(record['Data'])
    print(f"SeqNo: {record['SequenceNumber'][:20]}... | Data: {data}")

# Next iterator for continued polling
next_iterator = records_response['NextShardIterator']

In [ ]:
# Create a Firehose delivery stream to S3
firehose.create_delivery_stream(
    DeliveryStreamName='clickstream-to-s3',
    DeliveryStreamType='KinesisStreamAsSource',
    KinesisStreamSourceConfiguration={
        'KinesisStreamARN': 'arn:aws:kinesis:us-east-1:123456789012:stream/clickstream',
        'RoleARN':          'arn:aws:iam::123456789012:role/FirehoseRole'
    },
    ExtendedS3DestinationConfiguration={
        'RoleARN':    'arn:aws:iam::123456789012:role/FirehoseRole',
        'BucketARN':  'arn:aws:s3:::my-data-lake',
        'Prefix':     'clickstream/year=!{timestamp:yyyy}/month=!{timestamp:MM}/day=!{timestamp:dd}/',
        'BufferingHints': {
            'SizeInMBs':         64,    # deliver when buffer reaches 64 MB
            'IntervalInSeconds': 300    # or every 5 minutes, whichever first
        },
        'CompressionFormat': 'GZIP',
        # Optional Lambda transform
        'ProcessingConfiguration': {
            'Enabled': True,
            'Processors': [{
                'Type': 'Lambda',
                'Parameters': [{
                    'ParameterName':  'LambdaArn',
                    'ParameterValue': 'arn:aws:lambda:us-east-1:123456789012:function:transform'
                }]
            }]
        }
    }
)
print("Firehose delivery stream created")

In [ ]:
# Scale a provisioned stream by splitting a hot shard
# (doubles capacity of that shard by splitting into two)
kinesis.split_shard(
    StreamName='clickstream',
    ShardToSplit=shard_id,
    NewStartingHashKey='170141183460469231731687303715884105728'  # midpoint
)
print("Shard split initiated — stream now has 5 shards")

# Monitor stream metrics via CloudWatch:
# - IncomingBytes / IncomingRecords: producer throughput
# - GetRecords.IteratorAgeMilliseconds: consumer lag (high = consumer too slow)
# - WriteProvisionedThroughputExceeded: producer throttling (add shards)

## Summary

| Concept | Key Takeaway |
|---|---|
| Kinesis Data Streams | Real-time stream; shards; multiple consumers; data retained 1–365 days |
| Shard capacity | 1 MB/s or 1,000 records/s write; 2 MB/s read (shared) |
| Partition key | Determines shard; use high-cardinality keys to avoid hot shards |
| Standard consumer | Shared 2 MB/s per shard; poll with GetRecords |
| Enhanced Fan-Out | Dedicated 2 MB/s per shard per consumer; push via HTTP/2; ~70ms latency |
| Retention | 24h default; up to 365 days; consumers can replay |
| On-demand mode | Auto-scales; pay per GB; no shard management |
| Kinesis Firehose | Managed delivery to S3/Redshift/OpenSearch; near real-time; no retention |
| Firehose buffering | Delivers when buffer size or interval reached (60s minimum) |
| Firehose transform | Optional Lambda to filter/enrich/convert records before delivery |
| Kinesis Analytics | SQL or Flink on streaming data; windowed aggregations; real-time metrics |
| KDS vs Firehose | KDS: custom real-time processing; Firehose: managed delivery to destinations |
| KDS vs SQS | KDS: multiple consumers, replay, ordered per shard; SQS: one consumer per message |
| MSK | Managed Apache Kafka; Kafka-compatible API; topics/partitions/brokers |
| MSK Serverless | Auto-scales Kafka; no broker management |
| MSK Connect | Managed Kafka Connect; move data between Kafka and AWS services |
| KDS vs MSK | KDS: AWS-native; MSK: existing Kafka ecosystem or Kafka-specific features |
| IteratorAgeMilliseconds | CloudWatch metric for consumer lag — high value means consumers are falling behind |